# MV Independent Validation — Performance A/B/C

## 问题

LightGBM 原始输出是 row-level，一条 row 通常对应 `login_id + acct_nbr + date`。同一账户、员工或员工—账户组合可能有多条 rows，因此原始 Recall/FPR 会给重复较多的 entity 更大权重。

本 notebook 按三个层级去重评估：

- **A：Unique Account** — `acct_nbr`
- **B：Unique Employee** — `login_id`
- **C：Unique Employee-Account** — `login_id + acct_nbr`

聚合规则：

- `actual_label = max(all row labels)`
- `aggregated_score = max(all row probabilities)`
- `aggregated_prediction = 1` if `aggregated_score >= threshold`

该规则回答“这个 entity 是否至少被成功 flag 过一次”。

本 notebook只读取已保存的 model 和 feature tables，不 retrain、不调参、不永久保存，也不会覆盖 MD artifact。


## 1. Imports

In [ ]:
import numpy as np
import pandas as pd
import pyspark.sql.functions as F
from sklearn.metrics import confusion_matrix, roc_auc_score

pd.set_option("display.max_columns", 200)
pd.set_option("display.float_format", lambda x: f"{x:,.6f}")

## 2. Configuration

In [ ]:
IN_TIME_FEATURE_PATH = (
    "abfss://ml-artifact-insider-us@dsapdafazprdadls1.dfs.core.windows.net/"
    "ins_us_nms/v1/output/insider_us_nms_features_table_v2"
)
OOT_FEATURE_PATH = (
    "abfss://ml-artifact-insider-us@dsapdafazprdadls1.dfs.core.windows.net/"
    "ins_us_nms/v1/output/insider_us_nms_features_table_6m_v2"
)

MODEL_ARTIFACT_NAME = "lgbm_model.pkl"
MODEL_ARTIFACT_VERSION = 1
MODEL_ARTIFACT_TYPE = "model"
MODEL_TMP_PATH = "tmp/insider_us_nms"

TARGET = "label"
RECALCULATE_THRESHOLD = True
THRESHOLD_GRID = np.round(np.arange(0.50, 0.561, 0.001), 3)
MAX_VALIDATION_FPR = 0.44
FIXED_THRESHOLD = 0.50

## 3. Load MD utility functions

只加载 helper functions，不执行 save/write。


In [ ]:
%run ../utils/load_functions_utils

## 4. Exact model feature list

In [ ]:
FEATURES = [
    "inquiries", "maintenances", "prop_maintenances", "prop_inquiries",
    "after_hours_touches", "emp_prop_acct_seqs", "avg_time_bw_touch_dt",
    "mean_acct_balance_touch", "change_acct_balance", "num_status_changes",
    "mean_seq_len_in_time", "stddev_acct_balance", "avg_emp_touches_per_acct",
    "mean_acct_balance_touch_deviation", "emp_prop_touch_seq_ratio",
    "stddev_acct_balance_emp", "acct_prop_touch_seq_ratio",
    "num_contact_info_changes", "avg_emp_behavior_deviation",
    "num_address_changes"
]

REQUIRED_METADATA = [
    "login_id", "acct_nbr", "date", "lookback_window_start",
    "fraud_date", "label"
]
OPTIONAL_METADATA = [
    "insider_label", "label_split", "cv_fold", "job_family_description"
]

print("Feature count:", len(FEATURES))

## 5. Load saved tables

输出 in-time 和 OOT row counts。此处仍是原始 row-level 数据。


In [ ]:
features_spk = spark.read.format("delta").load(IN_TIME_FEATURE_PATH)
oot_spk = spark.read.format("delta").load(OOT_FEATURE_PATH)

print("In-time rows:", features_spk.count())
print("OOT rows:", oot_spk.count())

## 6. Required-column checks

In [ ]:
def validate_required_columns(df, required, name):
    missing = [x for x in required if x not in df.columns]
    if missing:
        raise ValueError(f"{name} missing columns: {missing}")
    print(f"{name}: required columns available.")

validate_required_columns(
    features_spk,
    REQUIRED_METADATA + FEATURES + ["label_split"],
    "In-time table"
)
validate_required_columns(
    oot_spk,
    REQUIRED_METADATA + FEATURES,
    "OOT table"
)

## 7. Recreate validation and test samples

不重新 split，只读取已保存的 `label_split`。Validation 仅用于 threshold selection。


In [ ]:
val_spk = features_spk.filter(F.col("label_split") == "val")
test_spk = features_spk.filter(F.col("label_split") == "test")

print("Validation rows:", val_spk.count())
print("Test rows:", test_spk.count())
print("OOT rows:", oot_spk.count())

## 8. Convert required data to pandas

In [ ]:
def available_columns(df, requested):
    return [x for x in requested if x in df.columns]

selected = REQUIRED_METADATA + OPTIONAL_METADATA + FEATURES

val_pd = val_spk.select(*available_columns(val_spk, selected)).toPandas()
test_pd = test_spk.select(*available_columns(test_spk, selected)).toPandas()
oot_pd = oot_spk.select(*available_columns(oot_spk, selected)).toPandas()

for df in [val_pd, test_pd, oot_pd]:
    for col in ["date", "lookback_window_start", "fraud_date"]:
        if col in df.columns:
            df[col] = pd.to_datetime(df[col])

print("Validation:", val_pd.shape)
print("Test:", test_pd.shape)
print("OOT:", oot_pd.shape)

## 9. Load saved MD LightGBM model

只创建内存对象，不修改 saved model。


In [ ]:
final_model = load_artifact_as_file(
    MODEL_ARTIFACT_NAME,
    MODEL_ARTIFACT_VERSION,
    MODEL_ARTIFACT_TYPE,
    tmp_file_path=MODEL_TMP_PATH
)

print("Model type:", type(final_model))

if hasattr(final_model, "feature_name_"):
    model_features = list(final_model.feature_name_)
    print("Feature order matches:", model_features == FEATURES)
    if model_features != FEATURES:
        raise ValueError(
            "Saved model feature order does not match FEATURES."
        )

## 10. Generate row-level probabilities

`model_probability` 是每条 employee-account-date row 的 `label=1` 概率。


In [ ]:
for df in [val_pd, test_pd, oot_pd]:
    df["model_probability"] = final_model.predict_proba(
        df[FEATURES]
    )[:, 1]

display(
    test_pd[
        ["login_id", "acct_nbr", "date", TARGET, "model_probability"]
    ].head(10)
)

## 11. Recreate operating threshold

候选 threshold 中，只保留 validation FPR ≤ 0.44；然后选 Recall 最高者。若 Recall 相同，优先更低 FPR，再优先更高 threshold。选出的 threshold 原样用于 Test 和 OOT。


In [ ]:
def safe_divide(a, b):
    return a / b if b != 0 else np.nan

def counts(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(
        y_true, y_pred, labels=[0, 1]
    ).ravel()
    return int(tn), int(fp), int(fn), int(tp)

def threshold_table(y_true, probabilities, thresholds):
    output = []
    for threshold in thresholds:
        pred = (probabilities >= threshold).astype(int)
        tn, fp, fn, tp = counts(y_true, pred)
        output.append({
            "threshold": float(threshold),
            "recall": safe_divide(tp, tp + fn),
            "fpr": safe_divide(fp, fp + tn),
            "precision": safe_divide(tp, tp + fp),
            "tn": tn, "fp": fp, "fn": fn, "tp": tp
        })
    return pd.DataFrame(output)

if RECALCULATE_THRESHOLD:
    validation_threshold_results = threshold_table(
        val_pd[TARGET].astype(int).to_numpy(),
        val_pd["model_probability"].to_numpy(),
        THRESHOLD_GRID
    )

    eligible = validation_threshold_results[
        validation_threshold_results["fpr"] <= MAX_VALIDATION_FPR
    ].copy()

    if eligible.empty:
        raise ValueError("No threshold satisfies validation FPR limit.")

    selected_threshold_row = (
        eligible
        .sort_values(
            ["recall", "fpr", "threshold"],
            ascending=[False, True, False]
        )
        .iloc[0]
    )
    selected_threshold = float(selected_threshold_row["threshold"])
    print("Selected threshold:", selected_threshold)
    display(pd.DataFrame([selected_threshold_row]))
else:
    selected_threshold = float(FIXED_THRESHOLD)
    print("Fixed threshold:", selected_threshold)

## 12. Create row-level 0/1 predictions

In [ ]:
for df in [test_pd, oot_pd]:
    df["model_prediction"] = (
        df["model_probability"] >= selected_threshold
    ).astype(int)

## 13. Metrics

- Recall：实际 positive entity 中被捕获的比例
- FNR：实际 positive entity 中被漏掉的比例
- FPR：实际 negative entity 中被误报的比例
- Precision：被 flag entity 中真正 positive 的比例
- Predicted-positive rate：最终 review volume 占比
- ROC AUC：基于 entity aggregated score 的排序能力


In [ ]:
def calculate_metrics(
    y_true, y_pred, probabilities, dataset, level
):
    y_true = np.asarray(y_true).astype(int)
    y_pred = np.asarray(y_pred).astype(int)
    probabilities = np.asarray(probabilities)

    tn, fp, fn, tp = counts(y_true, y_pred)
    total = tn + fp + fn + tp

    auc = np.nan
    if len(np.unique(y_true)) == 2:
        auc = roc_auc_score(y_true, probabilities)

    return pd.DataFrame([{
        "dataset": dataset,
        "analysis_level": level,
        "num_entities": total,
        "actual_positive": tp + fn,
        "actual_negative": tn + fp,
        "predicted_positive": tp + fp,
        "tp": tp, "fn": fn, "tn": tn, "fp": fp,
        "recall_tpr": safe_divide(tp, tp + fn),
        "false_negative_rate": safe_divide(fn, tp + fn),
        "false_positive_rate": safe_divide(fp, fp + tn),
        "precision": safe_divide(tp, tp + fp),
        "specificity_tnr": safe_divide(tn, tn + fp),
        "accuracy": safe_divide(tp + tn, total),
        "predicted_positive_rate": safe_divide(tp + fp, total),
        "roc_auc": auc,
        "threshold": selected_threshold
    }])

## 14. Entity aggregation

每个 entity 的真实标签和模型分数都取最大值。额外保留 source row count，便于判断重复 rows 是否放大原始 performance。


In [ ]:
def aggregate_entity(row_df, keys, dataset, level):
    result = (
        row_df
        .groupby(keys, dropna=False)
        .agg(
            actual_label=(TARGET, "max"),
            aggregated_score=("model_probability", "max"),
            num_source_rows=(TARGET, "size"),
            num_positive_rows=(TARGET, "sum"),
            first_observation_date=("date", "min"),
            last_observation_date=("date", "max")
        )
        .reset_index()
    )

    result["aggregated_prediction"] = (
        result["aggregated_score"] >= selected_threshold
    ).astype(int)

    metric = calculate_metrics(
        result["actual_label"],
        result["aggregated_prediction"],
        result["aggregated_score"],
        dataset,
        level
    )
    return result, metric

# A. Unique Account

Grouping key：`acct_nbr`

表示同一账户所有 rows 合并后，该账户是否至少被成功 flag 一次。它衡量 account coverage，但不能说明哪个员工造成风险。


In [ ]:
test_account, test_account_metric = aggregate_entity(
    test_pd, ["acct_nbr"], "In-Time Test", "A_Unique_Account"
)
oot_account, oot_account_metric = aggregate_entity(
    oot_pd, ["acct_nbr"], "OOT", "A_Unique_Account"
)

display(test_account_metric)
display(oot_account_metric)

# B. Unique Employee

Grouping key：`login_id`

同一员工跨账户、跨日期的所有 rows 合并后，只要任意 row 是 positive，则 employee actual label=1；只要任意 score 超过 threshold，则 employee 被视为 flagged。

这是最直接的 employee-oriented performance。


In [ ]:
test_employee, test_employee_metric = aggregate_entity(
    test_pd, ["login_id"], "In-Time Test", "B_Unique_Employee"
)
oot_employee, oot_employee_metric = aggregate_entity(
    oot_pd, ["login_id"], "OOT", "B_Unique_Employee"
)

display(test_employee_metric)
display(oot_employee_metric)

# C. Unique Employee-Account

Grouping key：`login_id + acct_nbr`

同一个 pair 跨日期和 rolling windows 的重复 rows 被合并。这是最接近当前 MD label unit 的去重 performance。


In [ ]:
test_pair, test_pair_metric = aggregate_entity(
    test_pd,
    ["login_id", "acct_nbr"],
    "In-Time Test",
    "C_Unique_Employee_Account"
)
oot_pair, oot_pair_metric = aggregate_entity(
    oot_pd,
    ["login_id", "acct_nbr"],
    "OOT",
    "C_Unique_Employee_Account"
)

display(test_pair_metric)
display(oot_pair_metric)

## 18. Main A/B/C output

重点解释：

- OOT Recall 下降：OOT capture 变弱
- OOT FPR 上升：误报和调查负担增加
- Employee 与 Pair 结果差异大：同一员工多账户结构影响明显
- Predicted-positive rate 高：需要 review 的 entity 比例较高


In [ ]:
abc_performance_summary = pd.concat(
    [
        test_account_metric, oot_account_metric,
        test_employee_metric, oot_employee_metric,
        test_pair_metric, oot_pair_metric
    ],
    ignore_index=True
)

display(abc_performance_summary)

## 19. Confusion matrices

布局：

```text
                 Predicted 0   Predicted 1
Actual 0              TN            FP
Actual 1              FN            TP
```


In [ ]:
def show_matrix(df, title):
    matrix = pd.crosstab(
        df["actual_label"],
        df["aggregated_prediction"],
        rownames=["Actual"],
        colnames=["Predicted"],
        dropna=False
    ).reindex(index=[0, 1], columns=[0, 1], fill_value=0)

    print(title)
    display(matrix)

show_matrix(test_account, "Test — A Unique Account")
show_matrix(oot_account, "OOT — A Unique Account")
show_matrix(test_employee, "Test — B Unique Employee")
show_matrix(oot_employee, "OOT — B Unique Employee")
show_matrix(test_pair, "Test — C Unique Employee-Account")
show_matrix(oot_pair, "OOT — C Unique Employee-Account")

## 20. False-negative lists

这些是实际 positive、但模型未 flag 的 unique entities，可用于 SME case review。


In [ ]:
def false_negatives(df):
    return (
        df.query(
            "actual_label == 1 and aggregated_prediction == 0"
        )
        .sort_values("aggregated_score", ascending=False)
    )

test_account_fn = false_negatives(test_account)
test_employee_fn = false_negatives(test_employee)
test_pair_fn = false_negatives(test_pair)

oot_account_fn = false_negatives(oot_account)
oot_employee_fn = false_negatives(oot_employee)
oot_pair_fn = false_negatives(oot_pair)

print("Test employee false negatives")
display(test_employee_fn)

print("Test employee-account false negatives")
display(test_pair_fn)

print("OOT employee false negatives")
display(oot_employee_fn)

print("OOT employee-account false negatives")
display(oot_pair_fn)

## 21. Row-level vs deduplicated performance

如果 row-level Recall 明显高于 A/B/C Recall，可能表示少数容易预测的 positive entities 产生了大量重复 rows，从而放大原始 performance。


In [ ]:
def row_metric(df, dataset):
    return calculate_metrics(
        df[TARGET],
        df["model_prediction"],
        df["model_probability"],
        dataset,
        "Original_Row_Level"
    )

row_vs_deduplicated = pd.concat(
    [
        row_metric(test_pd, "In-Time Test"),
        test_account_metric,
        test_employee_metric,
        test_pair_metric,
        row_metric(oot_pd, "OOT"),
        oot_account_metric,
        oot_employee_metric,
        oot_pair_metric
    ],
    ignore_index=True
)

display(row_vs_deduplicated)

## 22. In-time vs OOT change

所有 change 均为 `OOT - Test`：

- Recall change < 0：捕获能力下降
- FNR change > 0：漏报增加
- FPR change > 0：误报增加
- Precision change < 0：flag 准确性下降
- Flag-rate change > 0：review burden 增加


In [ ]:
def stability(test_metric, oot_metric):
    a = test_metric.iloc[0]
    b = oot_metric.iloc[0]
    return pd.DataFrame([{
        "analysis_level": a["analysis_level"],
        "test_recall": a["recall_tpr"],
        "oot_recall": b["recall_tpr"],
        "recall_change": b["recall_tpr"] - a["recall_tpr"],
        "test_fnr": a["false_negative_rate"],
        "oot_fnr": b["false_negative_rate"],
        "fnr_change": b["false_negative_rate"] - a["false_negative_rate"],
        "test_fpr": a["false_positive_rate"],
        "oot_fpr": b["false_positive_rate"],
        "fpr_change": b["false_positive_rate"] - a["false_positive_rate"],
        "test_precision": a["precision"],
        "oot_precision": b["precision"],
        "precision_change": b["precision"] - a["precision"],
        "test_flag_rate": a["predicted_positive_rate"],
        "oot_flag_rate": b["predicted_positive_rate"],
        "flag_rate_change": (
            b["predicted_positive_rate"] - a["predicted_positive_rate"]
        ),
        "test_auc": a["roc_auc"],
        "oot_auc": b["roc_auc"],
        "auc_change": b["roc_auc"] - a["roc_auc"]
    }])

temporal_stability = pd.concat(
    [
        stability(test_account_metric, oot_account_metric),
        stability(test_employee_metric, oot_employee_metric),
        stability(test_pair_metric, oot_pair_metric)
    ],
    ignore_index=True
)

display(temporal_stability)

## 23. Interpretation summary

- **A Account**：unique positive accounts 是否至少被 flag 一次
- **B Employee**：unique positive employees 是否至少被 flag 一次
- **C Employee-Account**：unique positive pairs 是否至少被 flag 一次

正式结论应称为：

```text
Deduplicated entity-level performance using maximum-score aggregation
```

不能称为 unique fraud-event Recall，因为这里没有按统一 fraud case/event ID 去重。

可继续使用的内存对象：

```text
abc_performance_summary
row_vs_deduplicated
temporal_stability
test_employee_fn
test_pair_fn
oot_employee_fn
oot_pair_fn
```
